# Homework: Uninformed Search

Students refer to pseudo codes of BFS, DFS, UCS, DLS, and IDS in [this link](https://drive.google.com/file/d/1q2LtrRCfemfiqyhfxNMcVJ3alvLh_pdV/view?usp=share_link) to implement the corresponding classes in TODO 1 - 5. \
Students can add supporting attributes and methods to the five classes of search strategies as needed.

# Libraries

In [1]:
import os
import heapq

# Graph class

In [2]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0

  def __str__(self): # toString()
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...

        # input.txt
        7 8
        0 1 5
        0 2 6
        1 3 12
        1 4 9
        2 5 5
        3 5 8
        3 6 7
        4 6 4
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for line in g:
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))

In [3]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 5), (2, 6)]
1: [(3, 12), (4, 9)]
2: [(5, 5)]
3: [(5, 8), (6, 7)]
4: [(6, 4)]



# Search Strategies

In [4]:
class SearchStrategy:
  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = [] # list of expanded vertices in the traversal order
    path = [] # path from src to dst
    return expanded, path

In [5]:
class BFS(SearchStrategy):
  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = []
    path = []

    frontier = [src]
    explored = []
    parent = {src: None}

    if src == dst:
        expanded.append(src)
        path.append(src)
        return expanded, path

    while frontier:
        node = frontier.pop(0)
        expanded.append(node)
        explored.append(node)

        children = sorted(g.AL.get(node, []), key=lambda x: x[0])

        for child, weight in children:
            if child not in explored and child not in frontier:
                parent[child] = node

                if child == dst:
                    curr = child
                    while curr is not None:
                        path.append(curr)
                        curr = parent[curr]
                    path.reverse()
                    return expanded, path

                frontier.append(child)

    return expanded, path

In [6]:
class DFS(SearchStrategy):
  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = []
    path = []

    frontier = [src]
    explored = []
    parent = {src: None}

    while frontier:
        node = frontier.pop()

        if node in explored:
            continue

        expanded.append(node)
        explored.append(node)

        if node == dst:
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expanded, path

        children = sorted(g.AL.get(node, []), key=lambda x: x[0], reverse=True)

        for child, weight in children:
            if child not in explored:
                parent[child] = node
                frontier.append(child)

    return expanded, path

In [7]:
class UCS(SearchStrategy):
  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = []
    path = []

    frontier = []
    heapq.heappush(frontier, (0, src))
    explored = []
    parent = {src: None}
    cost_so_far = {src: 0}

    while frontier:
        node_cost, node = heapq.heappop(frontier)

        if node in explored:
            continue

        expanded.append(node)
        explored.append(node)

        if node == dst:
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expanded, path

        children = sorted(g.AL.get(node, []), key=lambda x: x[0])
        for child, weight in children:
            child_cost = node_cost + weight
            in_frontier = any(child == item[1] for item in frontier)

            if child not in explored and not in_frontier:
                cost_so_far[child] = child_cost
                parent[child] = node
                heapq.heappush(frontier, (child_cost, child))
            elif in_frontier and child_cost < cost_so_far.get(child, float('inf')):
                cost_so_far[child] = child_cost
                parent[child] = node

                frontier = [item for item in frontier if item[1] != child]
                heapq.heappush(frontier, (child_cost, child))
                heapq.heapify(frontier)

    return expanded, path

In [8]:
class DLS(SearchStrategy):
  def __init__(self, LIM: int):
    self.LIM = LIM

  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = []
    path = []

    def recursive_dls(node, current_limit, current_path):
        expanded.append(node)
        if node == dst:
            return current_path + [node]
        elif current_limit == 0:
            return 'cutoff'
        else:
            cutoff_occurred = False
            children = sorted(g.AL.get(node, []), key=lambda x: x[0])

            for child, weight in children:
                result = recursive_dls(child, current_limit - 1, current_path + [node])
                if result == 'cutoff':
                    cutoff_occurred = True
                elif result != 'failure':
                    return result

            return 'cutoff' if cutoff_occurred else 'failure'

    res = recursive_dls(src, self.LIM, [])
    if isinstance(res, list):
        path = res

    return expanded, path

In [9]:
class IDS(SearchStrategy):
  def __init__(self, MAX_LIM: int):
    self.MAX_LIM = MAX_LIM

  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = []
    path = []

    for depth in range(self.MAX_LIM + 1):
        current_expanded = []

        def recursive_dls(node, current_limit, current_path):
            current_expanded.append(node)
            if node == dst:
                return current_path + [node]
            elif current_limit == 0:
                return 'cutoff'
            else:
                cutoff_occurred = False
                children = sorted(g.AL.get(node, []), key=lambda x: x[0])

                for child, weight in children:
                    result = recursive_dls(child, current_limit - 1, current_path + [node])
                    if result == 'cutoff':
                        cutoff_occurred = True
                    elif result != 'failure':
                        return result

                return 'cutoff' if cutoff_occurred else 'failure'

        res = recursive_dls(src, depth, [])
        expanded.extend(current_expanded)

        if isinstance(res, list):
            path = res
            return expanded, path

    return expanded, path

# Evaluation

In [10]:
bfs = BFS()
dfs = DFS()
ucs = UCS()
dls = DLS(LIM=3)
ids = IDS(MAX_LIM=5)

for stg in [bfs, dfs, ucs, dls, ids]:
  print(stg)
  expanded, path = stg.search(g, 0, g.V-1)
  print(expanded)
  print(path)

[0, 1, 2, 3]
[0, 1, 3, 6]
[0, 1, 3, 5, 6]
[0, 1, 3, 6]
[0, 1, 2, 5, 4, 3, 6]
[0, 1, 4, 6]
[0, 1, 3, 5, 6]
[0, 1, 3, 6]
[0, 0, 1, 2, 0, 1, 3, 4, 2, 5, 0, 1, 3, 5, 6]
[0, 1, 3, 6]


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec04-UninformedSearch-HW.ipynb**
*   Finally, submit the file

